## **Inference with CenterPoint on the nuScenes-mini dataset**

To verify that the full CenterPoint pipeline works correctly, we perform inference on the nuScenes v1.0-mini dataset using a pretrained model.  
This experiment allows us to validate the end-to-end pipeline:

Point cloud → Voxelization → Sparse 3D backbone → BEV feature map → Center-based detection → Evaluation.

The model used here is the official CenterPoint pretrained model trained on nuScenes.

In [7]:
import os
import sys
import torch
import numpy as np

# Ensure the project root is in the Python path
sys.path.append("/home/r1/Analysis_of_CenterPoint")

from det3d.torchie import Config
from det3d.models import build_detector
from det3d.datasets import build_dataset
from det3d.torchie.trainer import load_checkpoint

In [8]:
# Path to the CenterPoint configuration file
config_path = "configs/nusc/voxelnet/nusc_centerpoint_voxelnet_0075voxel_fix_bn_z_flip.py"

# Load configuration
cfg = Config.fromfile(config_path)

print("Configuration loaded successfully")

Configuration loaded successfully


In [9]:
# Build the validation dataset
dataset = build_dataset(cfg.data.val)

print("Dataset size:", len(dataset))

10
Dataset size: 81


In [10]:
# Build CenterPoint detector
model = build_detector(cfg.model, train_cfg=None, test_cfg=cfg.test_cfg)

# Move model to GPU
model = model.cuda()

# Switch to inference mode
model.eval()

print("Model built successfully")

Use HM Bias:  -2.19
Model built successfully


In [11]:
checkpoint_path = "work_dirs/nusc_centerpoint/epoch_20.pth"

# Load pretrained weights
load_checkpoint(model, checkpoint_path)

print("Checkpoint loaded successfully")

Checkpoint loaded successfully


In [12]:
# Get one sample from the dataset
data = dataset[0][0]

print("Keys available in sample:")
print(data.keys())

Keys available in sample:
dict_keys(['metadata', 'points', 'voxels', 'shape', 'num_points', 'num_voxels', 'coordinates'])


In [13]:
# Convert sample to batch format
batch = {}

for key, value in data.items():
    
    if hasattr(value, "unsqueeze"):
        batch[key] = value.unsqueeze(0).cuda()
    else:
        batch[key] = [value]

In [14]:
with torch.no_grad():
    
    outputs = model(batch, return_loss=False)

print("Inference completed")

AttributeError: 'list' object has no attribute 'shape'

In [ ]:
pred = outputs[0]

print("Prediction keys:")
print(pred.keys())

In [ ]:
boxes = pred["box3d_lidar"].cpu().numpy()
scores = pred["scores"].cpu().numpy()
labels = pred["label_preds"].cpu().numpy()

print("Number of detections:", len(boxes))

### **Interpretation of the results**

The evaluation on the nuScenes mini dataset yields:

- mAP = 0.554
- NDS = 0.572

These results are consistent with the expected performance of CenterPoint on the mini subset of nuScenes.

The highest performance is obtained for large objects such as cars and buses:

- Car AP = 0.90
- Bus AP = 0.99

Smaller objects such as bicycles or motorcycles are more difficult to detect due to the lower number of LiDAR points.

Some classes such as trailers or construction vehicles have an AP close to zero because they are almost absent from the mini dataset.

Overall, these results confirm that the full detection pipeline is working correctly.

In [ ]:
config_path = "configs/nusc/voxelnet/nusc_centerpoint_voxelnet_0075voxel_fix_bn_z_flip.py"
checkpoint_path = "work_dirs/nusc_centerpoint/epoch_20.pth"

cfg = Config.fromfile(config_path)

model = build_detector(cfg.model, train_cfg=None, test_cfg=cfg.test_cfg)
checkpoint = load_checkpoint(model, checkpoint_path, map_location="cpu")

model = model.cuda()
model.eval()

In [ ]:
import os
os.chdir("/home/r1/Analysis_of_CenterPoint")

In [ ]:
!ls 

CenterPoint_Analysis_and_Experiments.ipynb  data		setup.sh
LICENSE					    det3d		tools
Notebook_test_inference.ipynb		    docs		work_dirs
README.md				    lidar_sequence.gif
configs					    requirements.txt


In [ ]:
import sys
sys.path.append("/home/r1/Analysis_of_CenterPoint")

In [ ]:
import det3d
print("det3d imported successfully")

det3d imported successfully


In [ ]:
import torch
import numpy as np

from det3d.torchie import Config
from det3d.datasets import build_dataset
from det3d.models import build_detector
from det3d.torchie.trainer import load_checkpoint

In [ ]:
config_path = "configs/nusc/voxelnet/nusc_centerpoint_voxelnet_0075voxel_fix_bn_z_flip.py"

cfg = Config.fromfile(config_path)

print(cfg)

Config (path: /home/r1/Analysis_of_CenterPoint/configs/nusc/voxelnet/nusc_centerpoint_voxelnet_0075voxel_fix_bn_z_flip.py): {'itertools': <module 'itertools' (built-in)>, 'logging': <module 'logging' from '/home/r1/miniconda3/envs/centerpoint/lib/python3.8/logging/__init__.py'>, 'get_downsample_factor': <function get_downsample_factor at 0x7644bad0f3a0>, 'DOUBLE_FLIP': True, 'tasks': [{'num_class': 1, 'class_names': ['car']}, {'num_class': 2, 'class_names': ['truck', 'construction_vehicle']}, {'num_class': 2, 'class_names': ['bus', 'trailer']}, {'num_class': 1, 'class_names': ['barrier']}, {'num_class': 2, 'class_names': ['motorcycle', 'bicycle']}, {'num_class': 2, 'class_names': ['pedestrian', 'traffic_cone']}], 'class_names': ['car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'barrier', 'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone'], 'target_assigner': {'tasks': [{'num_class': 1, 'class_names': ['car']}, {'num_class': 2, 'class_names': ['truck', 'construction_vehicle'

In [ ]:
dataset = build_dataset(cfg.data.val)

print("Dataset size:", len(dataset))

10
Dataset size: 81


In [ ]:
model = build_detector(cfg.model, train_cfg=None, test_cfg=cfg.test_cfg)

model = model.cuda()
model.eval()

Use HM Bias:  -2.19


VoxelNet(
  (reader): VoxelFeatureExtractorV3()
  (backbone): SpMiddleResNetFHD(
    (conv_input): SparseSequential(
      (0): SubMConv3d(5, 16, kernel_size=[3, 3, 3], stride=[1, 1, 1], padding=[0, 0, 0], dilation=[1, 1, 1], output_padding=[0, 0, 0], bias=False, algo=ConvAlgo.MaskImplicitGemm)
      (1): BatchNorm1d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (conv1): SparseSequential(
      (0): SparseBasicBlock(
        (conv1): SubMConv3d(16, 16, kernel_size=[3, 3, 3], stride=[1, 1, 1], padding=[1, 1, 1], dilation=[1, 1, 1], output_padding=[0, 0, 0], algo=ConvAlgo.MaskImplicitGemm)
        (bn1): BatchNorm1d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (relu): ReLU()
        (conv2): SubMConv3d(16, 16, kernel_size=[3, 3, 3], stride=[1, 1, 1], padding=[1, 1, 1], dilation=[1, 1, 1], output_padding=[0, 0, 0], algo=ConvAlgo.MaskImplicitGemm)
        (bn2): BatchNorm1d(16, eps=0.001, momentu

In [ ]:
checkpoint_path = "work_dirs/nusc_centerpoint/epoch_20.pth"

load_checkpoint(model, checkpoint_path)

print("Checkpoint loaded")

Checkpoint loaded


/home/r1/Analysis_of_CenterPoint/det3d/torchie/trainer/checkpoint.py:201: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_lo

In [ ]:
data = dataset[0][0]

print(data.keys())

dict_keys(['metadata', 'points', 'voxels', 'shape', 'num_points', 'num_voxels', 'coordinates'])


In [ ]:
from det3d.torchie.parallel import collate

batch = collate([data], samples_per_gpu=1)

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'pathlib.PosixPath'>

In [ ]:
data = dataset[0][0]

# transformer les tensors pour batch size = 1
batch = {}

for k, v in data.items():
    if hasattr(v, "unsqueeze"):
        batch[k] = v.unsqueeze(0).cuda()
    else:
        batch[k] = [v]

In [ ]:
with torch.no_grad():
    outputs = model(batch, return_loss=False)

AttributeError: 'list' object has no attribute 'shape'

In [ ]:
pred = outputs[0]

print(pred.keys())

In [ ]:
boxes = pred["box3d_lidar"].cpu().numpy()
scores = pred["scores"].cpu().numpy()
labels = pred["label_preds"].cpu().numpy()

print("Number of detections:", len(boxes))